# Aura Healthcare Analytics - Capstone Project

Develop an end-to-end data analysis solution using Python for a healthcare dataset. The goal is to extract insights about patient visits, demographics, and regional trends to support Aura's data-driven healthcare decision-making.

1. **Import and Clean Data** Load the dataset, inspect for quality issues, handle missing values and duplicates, and optimize data types.
2.  **Data Processing and Statistical Analysis** Apply transformations to Age and Income, generate descriptive statistics, and compare results against the raw dataset.
3.  **Data Analysis with Pandas** Identify categorical and numerical variables, perform pivots and group-bys, and encode categorical data.
4.  **Data Visualization** Use matplotlib and seaborn to surface patterns through bar plots, heatmaps, and box plots.



---

# Task 1: Import and Clean Data

**Goal:** Load the NSMES1988 dataset, inspect its structure, identify and handle any data quality issues, optimize memory usage, and export a cleaned version.

In [2]:
import pandas as pd
import numpy as np

print(f"pandas version: {pd.__version__}")
print(f"numpy version:  {np.__version__}")


pandas version: 2.2.3
numpy version:  1.26.4


In [3]:
df = pd.read_csv("../../../data/NSMES1988.csv")

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")



Dataset loaded: 4,406 rows × 19 columns


In [4]:
df.head()

,Unnamed: 0,visits,nvisits,ovisits,novisits,emergency,hospital,health,chronic,adl,region,age,gender,married,school,income,employed,insurance,medicaid
0,1,5,0,0,0,0,1,average,2,normal,other,6.9,male,yes,6,2.8810,yes,yes,no
1,2,1,0,2,0,2,0,average,2,normal,other,7.4,female,yes,10,2.7478,no,yes,no
2,3,13,0,0,0,3,3,poor,4,limited,other,6.6,female,no,10,0.6532,no,no,yes
3,4,16,0,5,0,1,1,poor,2,limited,other,7.6,male,yes,3,0.6588,no,yes,no
4,5,3,0,0,0,0,0,average,2,limited,other,7.9,female,yes,6,0.6588,no,yes,no


In [5]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)

Shape: (4406, 19)

Columns: ['Unnamed: 0', 'visits', 'nvisits', 'ovisits', 'novisits', 'emergency', 'hospital', 'health', 'chronic', 'adl', 'region', 'age', 'gender', 'married', 'school', 'income', 'employed', 'insurance', 'medicaid']

Data types:
Unnamed: 0      int64
visits          int64
nvisits         int64
ovisits         int64
novisits        int64
emergency       int64
hospital        int64
health         object
chronic         int64
adl            object
region         object
age           float64
gender         object
married        object
school          int64
income        float64
employed       object
insurance      object
medicaid       object
dtype: object


In [7]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4406 entries, 0 to 4405
Data columns (total 19 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  4406 non-null   int64  
 1   visits      4406 non-null   int64  
 2   nvisits     4406 non-null   int64  
 3   ovisits     4406 non-null   int64  
 4   novisits    4406 non-null   int64  
 5   emergency   4406 non-null   int64  
 6   hospital    4406 non-null   int64  
 7   health      4406 non-null   object 
 8   chronic     4406 non-null   int64  
 9   adl         4406 non-null   object 
 10  region      4406 non-null   object 
 11  age         4406 non-null   float64
 12  gender      4406 non-null   object 
 13  married     4406 non-null   object 
 14  school      4406 non-null   int64  
 15  income      4406 non-null   float64
 16  employed    4406 non-null   object 
 17  insurance   4406 non-null   object 
 18  medicaid    4406 non-null   object 
dtypes: float64(2), int64(9), ob

### Initial inspection observations

- The dataset contians **4,406 rows and 19 columns**
- The project brief mentions 5 key features (Age, Income, Health, Region, Visits), but the actual dataset has 14 additional columns covering visit subcategories, demographics, health status, and insurance information.
- There is an extraneous 'Unnamed: 0' column; an artifact of a previous CSV export where the row index was saved as a column. This carries no analytical value and will be removed.
- All 19 columns have 4,406 non-null values, meaning **no missing data**.
- Several 'object' -dtype columns (e.g., 'health', 'region', 'gender') have only 204 unique values and are candidates for conversion to 'category' dtype.
- Baseline memory usage: **2.2 MB**.

In [8]:
df.isnull().sum()

missing_counts = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_pct": missing_pct
})
missing_summary

,missing_count,missing_pct
Unnamed: 0,0,0.0
visits,0,0.0
nvisits,0,0.0
ovisits,0,0.0
novisits,0,0.0
emergency,0,0.0
hospital,0,0.0
health,0,0.0
chronic,0,0.0
adl,0,0.0


### Missing Values report

The dataset contains **zero missing values** across all 19 columns. No imputation or row dropping was required. 

**Handling techniques that would have been applied (had missing values been present)** 

| Variable type | Technique | Rationale |
|---|---|---|
| Numeric (e.g., 'income', 'age') | Median imputation | Robust to skew; income is right-skewed |
| Count (e.g., 'visits', 'chronic') | Median or zero-fill | Counts often have natural zeros |
| Categorical (e.g., 'health', 'region') | Mode imputation | Fills with the most common category |
| Any column, <5% missing & random | Row drop ('dropna') | Minimal information loss |
| Any column, >30% missing | Drop the column | Imputation introduces too much bias |

In [9]:
dup_count = df.duplicated().sum()
print(f"Duplicate rows: {dup_count}")

if dup_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"After dropping duplicates: {df.shape[0]:,} rows")
else:
    print("No duplicates to remove.")

Duplicate rows: 0
No duplicates to remove.


### Duplicate records

The dataset contains **zero duplicate rows**. Since each row represents a unique survey respondent, no duplicate removal was necessary. A conditional drop was included in the code to handle duplicates automatically if the dataset were updated in the future. 

In [10]:
object_cols = df.select_dtypes(include="object").columns.tolist()
print(f"Object-typed columns: {object_cols}\n")

for col in object_cols:
    uniques = df[col].unique()
    print(f"{col:12} ({len(uniques)} unique): {list(uniques)}")

Object-typed columns: ['health', 'adl', 'region', 'gender', 'married', 'employed', 'insurance', 'medicaid']

health       (3 unique): ['average', 'poor', 'excellent']
adl          (2 unique): ['normal', 'limited']
region       (4 unique): ['other', 'midwest', 'northeast', 'west']
gender       (2 unique): ['male', 'female']
married      (2 unique): ['yes', 'no']
employed     (2 unique): ['yes', 'no']
insurance    (2 unique): ['yes', 'no']
medicaid     (2 unique): ['no', 'yes']


### Categorical column inspection

All eight `object`-dtype columns have low cardinality (2–4 unique values):

- **Multi-class:** `health` (3: excellent / average / poor), `region` (4: midwest / 
  northeast / west / other)
- **Binary:** `adl` (normal / limited), `gender` (male / female), `married` (yes / no), 
  `employed` (yes / no), `insurance` (yes / no), `medicaid` (yes / no)

All eight will be converted to the `category` dtype for memory efficiency.

**A note on the binary columns:** The four yes/no columns (`married`, `employed`, 
`insurance`, `medicaid`) could alternatively be encoded as Boolean or 0/1 values. 
That encoding is deferred to **Task 3**, where preparing categorical variables for 
modeling is the explicit goal. Encoding to 0/1 is a lossy, hard-to-reverse 
transformation, so it is best applied late in the pipeline — keeping the data in 
its richest, most flexible form (`category`, with original labels intact) through 
the cleaning stage.

In [11]:
# Capture starting memory so we can measure the improvement later
mem_before = df.memory_usage(deep=True).sum() / 1024
print(f"Memory before optimization: {mem_before:,.2f} KB")

# Drop the extraneous index column left over from a prior CSV export
df = df.drop(columns=["Unnamed: 0"])
print(f"Shape after dropping 'Unnamed: 0': {df.shape}")

Memory before optimization: 2,210.86 KB
Shape after dropping 'Unnamed: 0': (4406, 18)


In [12]:
# Convert all object columns to category
categorical_cols = ["health", "adl", "region", "gender",
                    "married", "employed", "insurance", "medicaid"]
for col in categorical_cols:
    df[col] = df[col].astype("category")

# Downcast integer columns to the smallest type that fits their range
int_cols = ["visits", "nvisits", "ovisits", "novisits",
            "emergency", "hospital", "chronic", "school"]
for col in int_cols:
    df[col] = pd.to_numeric(df[col], downcast="integer")

# Downcast float columns
df["age"] = pd.to_numeric(df["age"], downcast="float")
df["income"] = pd.to_numeric(df["income"], downcast="float")

print("Updated dtypes:")
print(df.dtypes)

Updated dtypes:
visits           int8
nvisits          int8
ovisits         int16
novisits        int16
emergency        int8
hospital         int8
health       category
chronic          int8
adl          category
region       category
age           float32
gender       category
married      category
school           int8
income        float32
employed     category
insurance    category
medicaid     category
dtype: object


In [13]:
mem_after = df.memory_usage(deep=True).sum() / 1024
reduction = (1 - mem_after / mem_before) * 100

print(f"Memory before: {mem_before:>10,.2f} KB")
print(f"Memory after:  {mem_after:>10,.2f} KB")
print(f"Reduction:     {reduction:>9.1f}%")

Memory before:   2,210.86 KB
Memory after:      113.90 KB
Reduction:          94.8%


### Data type optimization

The raw dataset used wide default dtypes throughout: eight text columns stored as 
generic `object`, all count columns as `int64`, and `age`/`income` as `float64`. 
Three optimizations were applied:

- **Object → category** for all eight low-cardinality text columns. Instead of 
  storing a full string per row, pandas stores each unique value once and uses 
  small integer codes per row. This produced the largest savings.
- **Integer downcasting** via `pd.to_numeric(downcast="integer")`, which auto-selects 
  the smallest fitting type. Most count columns fit in `int8`; `ovisits` and 
  `novisits` required `int16`.
- **Float downcasting** of `age` and `income` from `float64` to `float32`.

**Result: memory dropped from 2,210.86 KB to 113.90 KB — a 94.8% reduction**, with 
no loss of information.

On a dataset this small the absolute saving is negligible, but the same technique 
applied to a multi-gigabyte dataset can determine whether the data fits in memory 
at all. Efficient dtypes are the first optimization to reach for before moving to 
compressed or on-disk formats like Parquet or HDF5.

In [14]:
output_path = "../../../data/NSMES1988new.csv"
df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to: {output_path}")

# Verify by reading it back
verify = pd.read_csv(output_path)
print(f"Verification — re-read shape: {verify.shape}")

Saved cleaned dataset to: ../../../data/NSMES1988new.csv
Verification — re-read shape: (4406, 18)


### Task 1 Summary Report

#### Dataset overview
- **Source:** NSMES1988 — National Medical Expenditure Survey, 1988. A U.S. survey 
  of individuals aged 66 and older (Medicare-eligible population).
- **Size:** 4,406 records × 19 original columns (18 after cleaning).
- **Purpose:** Prepare the data for downstream machine learning — e.g., predicting 
  healthcare utilization (visits, hospital stays) from demographic and economic 
  variables.

#### Data quality findings
| Check | Result | Action taken |
|---|---|---|
| Missing values | 0 across all columns | None required |
| Duplicate rows | 0 | None required |
| Extraneous columns | `Unnamed: 0` (CSV index artifact) | Dropped |
| Encoded units | `age` stored in decades, `income` in $10,000s | Noted; rescaling deferred to Task 2 |

#### Transformations applied
1. Dropped the `Unnamed: 0` column (leftover row index from a prior export).
2. Converted eight low-cardinality text columns to `category` dtype.
3. Downcast count columns to `int8`/`int16` and `age`/`income` to `float32`.

#### Memory optimization
Reduced in-memory footprint from **2,210.86 KB to 113.90 KB (94.8% reduction)** 
with no loss of information.

#### Note on the output file
The cleaned data was exported to `NSMES1988new.csv` with `index=False` (avoiding the 
index-as-column artifact found in the original). Because CSV does not preserve dtype 
information, the optimized dtypes apply only in memory; reloading the file re-infers 
default types. A binary format such as Parquet would preserve dtypes if needed.

#### Output
- `data/NSMES1988new.csv` — cleaned dataset, ready as the input for Task 2.

#### Next steps (Task 2)
Rescale `age` (×10) and `income` (×10,000) into human-readable units, generate 
descriptive statistics with `describe()`, and compare against the raw scale.